In [ ]:
# ===============================
# IMPORT LIBRARIES
# ===============================
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===============================
# PRUNABLE LINEAR LAYER
# ===============================
class PrunableLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()

        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.01)
        self.bias = nn.Parameter(torch.zeros(out_features))

        # Small initialization (important)
        self.gate_scores = nn.Parameter(torch.randn(out_features, in_features) * 0.1)

    def forward(self, x):
        # Temperature scaling (important)
        gates = torch.sigmoid(self.gate_scores * 5)

        pruned_weights = self.weight * gates
        return torch.matmul(x, pruned_weights.t()) + self.bias


# ===============================
# MODEL
# ===============================
class SelfPruningNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = PrunableLinear(32*32*3, 512)
        self.fc2 = PrunableLinear(512, 256)
        self.fc3 = PrunableLinear(256, 10)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(x.size(0), -1)

        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)

        return x


# ===============================
# SPARSITY LOSS (STRONG VERSION)
# ===============================
def compute_sparsity_loss(model):
    loss = 0
    total = 0

    for module in model.modules():
        if isinstance(module, PrunableLinear):
            gates = torch.sigmoid(module.gate_scores * 5)

            # Strong push towards 0
            loss += torch.sum(gates)
            total += gates.numel()

    return loss / total


# ===============================
# DATA
# ===============================
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False)


# ===============================
# TRAINING
# ===============================
def train_model(lambda_val):
    model = SelfPruningNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    epochs = 10

    for epoch in range(epochs):
        model.train()

        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)

            ce_loss = criterion(outputs, labels)
            sparsity_loss = compute_sparsity_loss(model)

            loss = ce_loss + lambda_val * sparsity_loss

            loss.backward()
            optimizer.step()

        print(f"Lambda {lambda_val} | Epoch {epoch+1} done")

    return model


# ===============================
# EVALUATION
# ===============================
def evaluate_model(model):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total


# ===============================
# SPARSITY CALCULATION
# ===============================
def calculate_sparsity(model):
    total = 0
    zero = 0

    for module in model.modules():
        if isinstance(module, PrunableLinear):
            gates = torch.sigmoid(module.gate_scores * 5)

            total += gates.numel()
            zero += torch.sum(gates < 0.05).item()

    return 100 * zero / total


# ===============================
# RUN EXPERIMENT
# ===============================
lambdas = [0.1, 1.0, 10.0]
results = []
best_model = None

for lam in lambdas:
    print(f"\nTraining with lambda = {lam}")

    model = train_model(lam)

    acc = evaluate_model(model)
    sparsity = calculate_sparsity(model)

    print(f"Accuracy: {acc:.2f}% | Sparsity: {sparsity:.2f}%")

    results.append((lam, acc, sparsity))
    best_model = model


# ===============================
# RESULTS TABLE
# ===============================
print("\nFinal Results:")
print("Lambda | Accuracy | Sparsity")

for r in results:
    print(f"{r[0]} | {r[1]:.2f}% | {r[2]:.2f}%")


# ===============================
# GATE DISTRIBUTION PLOT
# ===============================
all_gates = []

for module in best_model.modules():
    if isinstance(module, PrunableLinear):
        gates = torch.sigmoid(module.gate_scores * 5).detach().cpu().numpy()
        all_gates.extend(gates.flatten())

plt.hist(all_gates, bins=50)
plt.title("Gate Value Distribution")
plt.xlabel("Gate Value")
plt.ylabel("Frequency")
plt.show()